In [1]:
import geopandas as gpd
import pandas as pd
from datetime import datetime, timedelta
import requests
import stackstac
import pystac_client
import os
import rasterio
from rasterio.vrt import WarpedVRT

In [ ]:
# Load the shapefile
shapefile_path = "../data/NFDB_poly_20210707_large_fires.shp"
gdf = gpd.read_file(shapefile_path)
gdf_latlon = gdf.to_crs(epsg=4326)

In [ ]:
# Download assets
download_dir = "../downloadedData/Sentinel2/"
os.makedirs(download_dir, exist_ok=True)

In [18]:
def make_fireComposite_VRT_withGDAL(swir22, nir8A, red,fireComposite_VRT):
    
    input_files = [
        swir22,
        nir8A,
        red
    ]

    output_vrt = fireComposite_VRT

    vrt_options = {
        "separate": True
    }


    gdal.BuildVRT(
        output_vrt,
        input_files,
        separate=True  # equivalent to QGIS SEPARATE=True
    )

In [19]:
# Make the request
def search_in_ogc_api(bbox, REP_DATE):
    start_date = f"2025-{REP_DATE:%m-%d}T00:00:00Z" #We dont have data in 2020, so artificial to do 2025!
    end_date = f"2025-{REP_DATE:%m-%d}T23:59:59Z"
    ogc_api_url = 'https://www-staging-eodms.aws.nrcan-rncan.cloud/maps/egs/wfs3'
    collection_id = 'sentinel2'
    datetime_range = f"{start_date}/{end_date}"
    search_url = f"{ogc_api_url}/collections/{collection_id}/items" 
    params = {
    "datetime": datetime_range, #capital D for ms4w - Datetime
     "bbox":",".join(map(str, bbox)),
     "limit":100
    }
    print("Starting to search for products")
    response = requests.get(search_url, params=params)
    desired_properties = ["producttype", "acquisition_start", "product_link"]
    # Check and parse the response
    if response.status_code == 200:
        features = response.json().get("features", [])
        #print(f"Found {len(features)} features between {start_date} and {end_date}.")
        print("Product Type","       ", "Acquisition start","          ", "Product Link", )
        for feature in features[:100]:      
            properties = feature.get("properties", {})
            product_type = properties.get('producttype')
            acquisition_start = properties.get('acquisition_start')
            product_link = properties.get('product_link')
            print(product_type,"       ", acquisition_start,"          ", product_link)
    else:
        print(f"Failed to retrieve features. Status code: {response.status_code}")


In [20]:
def search_in_stac_api(bbox, reported_date):
    stac_api_url = 'https://earth-search.aws.element84.com/v1'
    stac_start_date = f"{reported_date:%Y-%m-%d}T00:00:00Z"
    end_date = reported_date + timedelta(days=7)
    stac_end_date = f"{end_date:%Y-%m-%d}T23:59:59Z"
    print("starting to search in stac api url")
    print(f"bbox is: {bbox}")
    stac_catalog = pystac_client.Client.open(stac_api_url)
    results = stac_catalog.search(collections=["sentinel-2-l2a"], datetime=[stac_start_date,stac_end_date], bbox=bbox)
    for entry in results.items():
        print(entry.id)
    for item in results.items():
        asset_swir22 = item.assets["swir22"]  # for NBR
        asset_nir8A = item.assets["nir08"]  # for NBR
        asset_red = item.assets["red"]  # for NBR
        asset_rededge2 = item.assets["rededge2"] # for BAI2
        asset_rededge3 = item.assets["rededge3"] # for BAI2
        
        url_swir22 = asset_swir22.href
        url_nir8A = asset_nir8A.href
        url_red = asset_red.href
        url_rededge2 = asset_rededge2.href
        url_rededge3 = asset_rededge3.href
        
        filename_swir22 = os.path.join(download_dir, f"{item.id}_swir22.tif")
        filename_nir8A = os.path.join(download_dir, f"{item.id}_nir8A.tif")
        filename_red = os.path.join(download_dir, f"{item.id}_red.tif")
        filename_rededge2 = os.path.join(download_dir, f"{item.id}_rededge2.tif")
        filename_rededge3 = os.path.join(download_dir, f"{item.id}_rededge3.tif")
        
        response_swir22 = requests.get(url_swir22)
        response_nir8A = requests.get(url_nir8A)
        response_red = requests.get(url_red)
        response_rededge2 = requests.get(url_rededge2)
        response_rededge3 = requests.get(url_rededge3)
        
        print(f"Downloading {filename_swir22}...")
        with open(filename_swir22, "wb") as f:
            f.write(response_swir22.content)
        print(f"Downloading {filename_nir8A}...")
        with open(filename_nir8A, "wb") as f:
            f.write(response_nir8A.content)
        print(f"Downloading {filename_red}...")
        with open(filename_red, "wb") as f:
            f.write(response_red.content)
        with open(filename_rededge2, "wb") as f:
            f.write(response_rededge2.content)
        with open(filename_rededge3, "wb") as f:
            f.write(response_rededge3.content)
        
        #filename_fireComposite_VRT = os.path.join(download_dir, f"{item.id}_fireComposite.tif.vrt")
        #make_fireComposite_VRT_withGDAL(filename_swir22, filename_nir8A, filename_red, filename_fireComposite_VRT)
    print("Download complete.")


In [21]:
# Define your custom function
def process_polygon(polygon, SIZE_HA, SRC_AGENCY, REP_DATE, OUT_DATE):
    # Example operation: print area and attributes
    #print(f"Area: {polygon.area}")
    print(f"bbox: {polygon.bounds}")
    print(f"Src_Agency: {SRC_AGENCY} Size_ha: {SIZE_HA} Reported_Date: {REP_DATE} Out_Date: {OUT_DATE}")
    bbox = polygon.bounds
    
    search_in_stac_api(bbox, REP_DATE)
    #search_in_ogc_api(bbox, REP_DATE)
    # Add your custom logic here

In [22]:
# Iterate through each row (polygon) in the GeoDataFrame
count = 0
print(f"Processing wildfire polygons for year: 2000")
for idx, row in gdf_latlon.iterrows():
    geometry = row.geometry
    REP_YEAR = row['YEAR']
    if REP_YEAR == 2020:
        #print(f"YEAR:{REP_YEAR}")
        REP_DATE = row['REP_DATE']  # Replace with actual attribute name
        REP_DATE = REP_DATE.date()
        OUT_DATE = row['OUT_DATE'] # pd.isna doesnt work for NaT, is None, IS NULL, =='' doesnt work either, 
        OUT_DATE = OUT_DATE.date()
        SIZE_HA = row['SIZE_HA']
        SRC_AGENCY = row['SRC_AGENCY']        
        if SIZE_HA > 200 and SRC_AGENCY == 'BC':
            process_polygon(geometry, SIZE_HA, SRC_AGENCY, REP_DATE, OUT_DATE)
            count = count + 1
            #print(f"{count}: Src_Agency: {SRC_AGENCY} Size_ha: {SIZE_HA} Reported_Date: {REP_DATE} Out_Date: {OUT_DATE}")
print(f"Total number of polygons is:  {count}")

Processing wildfire polygons for year: 2000
bbox: (-104.68930263501665, 61.07289475148776, -104.64097648565951, 61.09300061795055)
Src_Agency: NT Size_ha: 278.107125962 Reported_Date: 2020-08-13 Out_Date: NaT
starting to search in stac api url
bbox is: (-104.68930263501665, 61.07289475148776, -104.64097648565951, 61.09300061795055)
S2B_13VEH_20200818_1_L2A
S2B_13VEH_20200818_0_L2A
S2A_13VEH_20200816_1_L2A
S2A_13VEH_20200816_0_L2A
S2A_13VEH_20200813_1_L2A
S2A_13VEH_20200813_0_L2A
Download complete.
bbox: (-116.8306263135738, 62.83774813966927, -116.7618177843304, 62.86749029043764)
Src_Agency: NT Size_ha: 656.833856461 Reported_Date: 2020-06-29 Out_Date: 2020-07-21
starting to search in stac api url
bbox is: (-116.8306263135738, 62.83774813966927, -116.7618177843304, 62.86749029043764)
S2A_11VMK_20200705_1_L2A
S2A_11VMK_20200705_0_L2A
S2A_11VNK_20200705_0_L2A
S2A_11VNK_20200705_1_L2A
S2B_11VMK_20200704_1_L2A
S2B_11VMK_20200704_0_L2A
S2B_11VNK_20200704_1_L2A
S2B_11VNK_20200704_0_L2A
S2A_

In [ ]:
def plot_RGB_cog(rgb):
    # Load each band from its respective COG file
    with rasterio.open(filename_swir22) as red_src:
        red = red_src.read(1)

    with rasterio.open(filename_nir) as green_src:
        green = green_src.read(1)

    with rasterio.open(filename_red) as blue_src:
        blue = blue_src.read(1)

    # Stack into RGB format
    rgb = np.stack([red, green, blue], axis=-1)
    
    # Normalize to [0, 1] for display
    rgb_normalized = rgb.astype(np.float32)
    rgb_normalized /= rgb_normalized.max()

    # Plot using matplotlib
    plt.figure(figsize=(10, 10))
    plt.imshow(rgb_normalized)
    plt.title("RGB Composite")
    plt.axis('off')
    plt.show()
